# Pilot 3 on the pod

Cell 1 on an RTX A6000 (14B and 8B). Cell 2 on a separate A100 80 GB pod (32B). Each cell beeps when it ends: three rising beeps = done, two low beeps = stopped (scroll up for the error). Every step resumes where it left off, so after a crash just run the same cell again.


In [ ]:
# ===== CELL 1 (RTX A6000): 14B and 8B, both stages and the goal module (about 30 minutes) =====
# Results are not shown. At the three rising beeps: download /workspace/pilot3_results_a6000.tar.gz,
# send it to Claude, and terminate this pod.
import os, subprocess, json, numpy as np
from IPython.display import Audio, display

REPO = "/workspace/llm-value-trade-offs"
P3 = f"{REPO}/pilot3-value-gate"
os.environ["HF_HOME"] = "/workspace/hf"          # models are cached on the volume disk

def beep(ok=True):
    """Three rising beeps = done. Two low beeps = something failed."""
    sr = 22050
    t = np.linspace(0, 0.22, int(sr * 0.22), False)
    gap = np.zeros(int(sr * 0.08))
    freqs = [523, 659, 784] if ok else [220, 165]
    display(Audio(np.concatenate([np.concatenate([0.5 * np.sin(2 * np.pi * f * t), gap]) for f in freqs]),
                  rate=sr, autoplay=True))

def run(cmd, log_file=None):
    """Run a shell command and show its output live (or send it to a file, unseen)."""
    print(f"\n>>> {cmd}", flush=True)
    if log_file:
        cmd = f"({cmd}) > {log_file} 2>&1"
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Failed: {cmd}" + (f" (see {log_file})" if log_file else ""))

def run_all(steps):
    try:
        for s in steps:
            run(*s) if isinstance(s, tuple) else run(s)
    except Exception as e:
        print(f"\n*** STOPPED: {e}")
        beep(False)
        return False
    beep(True)
    return True

SETUP = [
    f"cd /workspace && (test -d {REPO} || git clone https://github.com/JasperBlank/llm-value-trade-offs.git)",
    f"cd {REPO} && git fetch -q origin main && git reset -q --hard origin/main && git log -1 --format='Running commit %h: %s'",
    f"cd {P3} && pip install -q -r requirements.txt",
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader && df -h /workspace | tail -1",
]

run_all(SETUP + [
    f"cd {P3} && python run.py --model quick --quick",           # small model: is everything working?
    f"cd {P3} && python run.py --model 14b",
    f"cd {P3} && python run.py --model 8b",
    (f"cd {P3} && python analyze.py", "/workspace/analyze_a6000.log"),     # output goes to a file, unseen
    f"cd {P3}/results && tar czf /workspace/pilot3_results_a6000.tar.gz real /workspace/analyze_a6000.log && ls -la /workspace/pilot3_results_a6000.tar.gz",
])


In [ ]:
# ===== CELL 2 (A NEW pod: A100 80 GB, volume disk 100 GB): 32B (about 30-40 minutes) =====
# Results are not shown. At the three rising beeps: download /workspace/pilot3_results_32b.tar.gz,
# send it to Claude, and terminate this pod.
import os, subprocess, json, numpy as np
from IPython.display import Audio, display

REPO = "/workspace/llm-value-trade-offs"
P3 = f"{REPO}/pilot3-value-gate"
os.environ["HF_HOME"] = "/workspace/hf"          # models are cached on the volume disk

def beep(ok=True):
    """Three rising beeps = done. Two low beeps = something failed."""
    sr = 22050
    t = np.linspace(0, 0.22, int(sr * 0.22), False)
    gap = np.zeros(int(sr * 0.08))
    freqs = [523, 659, 784] if ok else [220, 165]
    display(Audio(np.concatenate([np.concatenate([0.5 * np.sin(2 * np.pi * f * t), gap]) for f in freqs]),
                  rate=sr, autoplay=True))

def run(cmd, log_file=None):
    """Run a shell command and show its output live (or send it to a file, unseen)."""
    print(f"\n>>> {cmd}", flush=True)
    if log_file:
        cmd = f"({cmd}) > {log_file} 2>&1"
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Failed: {cmd}" + (f" (see {log_file})" if log_file else ""))

def run_all(steps):
    try:
        for s in steps:
            run(*s) if isinstance(s, tuple) else run(s)
    except Exception as e:
        print(f"\n*** STOPPED: {e}")
        beep(False)
        return False
    beep(True)
    return True

SETUP = [
    f"cd /workspace && (test -d {REPO} || git clone https://github.com/JasperBlank/llm-value-trade-offs.git)",
    f"cd {REPO} && git fetch -q origin main && git reset -q --hard origin/main && git log -1 --format='Running commit %h: %s'",
    f"cd {P3} && pip install -q -r requirements.txt",
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader && df -h /workspace | tail -1",
]

run_all(SETUP + [
    f"cd {P3} && python run.py --model 32b",
    (f"cd {P3} && python analyze.py", "/workspace/analyze_32b.log"),
    f"cd {P3}/results && tar czf /workspace/pilot3_results_32b.tar.gz real /workspace/analyze_32b.log && ls -la /workspace/pilot3_results_32b.tar.gz",
])
